<a href="https://colab.research.google.com/github/vijayalakshmish/Agent/blob/main/Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
"""
Retail ontology agent (Gemini version) - hardened.

Colab:
  !pip install -U google-genai
  %run retail_agent.py              # interactive agent
  %run retail_agent.py --selftest   # no AI, no key needed

Terminal:
  pip install -U google-genai
  export GEMINI_API_KEY=AQ.... (or AIza...)     # free key: https://aistudio.google.com/apikey
  python retail_agent.py
"""
import json
import os
import sys

# ---------------------------------------------------------------- 1. DATA
DB = {
    "Supplier": {
        "SUP1": {"id": "SUP1", "name": "FreshFarm Foods", "lead_time_days": 2},
        "SUP2": {"id": "SUP2", "name": "TechGear Ltd", "lead_time_days": 7},
    },
    "Product": {
        "P1": {"id": "P1", "name": "Basmati Rice 5kg", "category": "grocery", "price": 650, "supplier_id": "SUP1"},
        "P2": {"id": "P2", "name": "Wireless Earbuds", "category": "electronics", "price": 2499, "supplier_id": "SUP2"},
        "P3": {"id": "P3", "name": "Phone Charger", "category": "electronics", "price": 799, "supplier_id": "SUP2"},
        "P4": {"id": "P4", "name": "Olive Oil 1L", "category": "grocery", "price": 899, "supplier_id": "SUP1"},
    },
    "Store": {
        "ST1": {"id": "ST1", "name": "Downtown"},
        "ST2": {"id": "ST2", "name": "Mall"},
        "ST3": {"id": "ST3", "name": "Airport"},
    },
    "Inventory": {  # one row = one product in one store
        "I1": {"id": "I1", "product_id": "P1", "store_id": "ST1", "quantity": 4, "reorder_point": 20},
        "I2": {"id": "I2", "product_id": "P1", "store_id": "ST2", "quantity": 80, "reorder_point": 20},
        "I3": {"id": "I3", "product_id": "P2", "store_id": "ST1", "quantity": 15, "reorder_point": 10},
        "I4": {"id": "I4", "product_id": "P2", "store_id": "ST3", "quantity": 3, "reorder_point": 10},
        "I5": {"id": "I5", "product_id": "P3", "store_id": "ST1", "quantity": 0, "reorder_point": 15},
        "I6": {"id": "I6", "product_id": "P4", "store_id": "ST2", "quantity": 120, "reorder_point": 20},
        "I7": {"id": "I7", "product_id": "P4", "store_id": "ST1", "quantity": 90, "reorder_point": 20},
    },
    "PurchaseOrder": {},
}
OBJECT_FIELDS = {
    "Supplier": ["id", "name", "lead_time_days"],
    "Product": ["id", "name", "category", "price", "supplier_id"],
    "Store": ["id", "name"],
    "Inventory": ["id", "product_id", "store_id", "quantity", "reorder_point"],
    "PurchaseOrder": ["id", "product_id", "supplier_id", "quantity", "status"],
}

# ---------------------------------------------------------------- 2. ONTOLOGY
# (from_type, link_name) -> (to_type, foreign_key, cardinality)
#   "one":  from_obj[fk] is the id of the target
#   "many": target_obj[fk] is the id of from_obj
LINKS = {
    ("Product", "supplier"): ("Supplier", "supplier_id", "one"),
    ("Product", "inventory"): ("Inventory", "product_id", "many"),
    ("Supplier", "products"): ("Product", "supplier_id", "many"),
    ("Store", "inventory"): ("Inventory", "store_id", "many"),
    ("Inventory", "product"): ("Product", "product_id", "one"),
    ("Inventory", "store"): ("Store", "store_id", "one"),
    ("PurchaseOrder", "product"): ("Product", "product_id", "one"),
    ("PurchaseOrder", "supplier"): ("Supplier", "supplier_id", "one"),
}


def _stock(product_id, store_id):
    for inv in DB["Inventory"].values():
        if inv["product_id"] == product_id and inv["store_id"] == store_id:
            return inv
    return None


# Action types: validated operations. The ONLY way the agent may change data.
def _create_purchase_order(product_id, quantity):
    quantity = int(quantity)
    prod = DB["Product"].get(product_id)
    if not prod:
        raise ValueError(f"Unknown product {product_id}")
    if not 1 <= quantity <= 1000:
        raise ValueError("Quantity must be between 1 and 1000")
    oid = f"PO{len(DB['PurchaseOrder']) + 1}"
    DB["PurchaseOrder"][oid] = {"id": oid, "product_id": product_id,
                                "supplier_id": prod["supplier_id"],
                                "quantity": quantity, "status": "sent"}
    return f"Created {oid}: {quantity} x {prod['name']} from supplier {prod['supplier_id']}"


def _transfer_stock(product_id, from_store_id, to_store_id, quantity):
    quantity = int(quantity)
    src, dst = _stock(product_id, from_store_id), _stock(product_id, to_store_id)
    if not src or not dst:
        raise ValueError("Product must be stocked in both stores")
    if quantity <= 0 or quantity > src["quantity"]:
        raise ValueError(f"Source store only has {src['quantity']} units")
    src["quantity"] -= quantity
    dst["quantity"] += quantity
    return f"Moved {quantity} of {product_id} from {from_store_id} to {to_store_id}"


def _apply_markdown(product_id, percent):
    percent = float(percent)
    prod = DB["Product"].get(product_id)
    if not prod:
        raise ValueError(f"Unknown product {product_id}")
    if not 5 <= percent <= 50:
        raise ValueError("Markdown must be between 5% and 50%")
    prod["price"] = round(prod["price"] * (1 - percent / 100), 2)
    return f"{prod['name']} price is now {prod['price']}"


ACTIONS = {
    "create_purchase_order": {
        "description": "Order more stock of a product from its supplier.",
        "params": {"product_id": "string", "quantity": "integer 1-1000"},
        "fn": _create_purchase_order,
    },
    "transfer_stock": {
        "description": "Move units of a product from one store to another.",
        "params": {"product_id": "string", "from_store_id": "string",
                   "to_store_id": "string", "quantity": "integer"},
        "fn": _transfer_stock,
    },
    "apply_markdown": {
        "description": "Reduce a product's price by a percent (5-50).",
        "params": {"product_id": "string", "percent": "number 5-50"},
        "fn": _apply_markdown,
    },
}
# All three change business data, so all require a human "y" first.


def human_approval(name, params):
    print(f"\n[APPROVAL NEEDED] {name} {json.dumps(params)}")
    return input("Approve? (y/n): ").strip().lower() == "y"


# ---------------------------------------------------------------- 3. TOOLS
# Plain functions with simple types (str) so Gemini can build a clean schema.
VERBOSE = False


def _log(name, **kwargs):
    if VERBOSE:
        print(f"  tool> {name}({json.dumps(kwargs)})")


def _match(obj, field, value):
    return str(obj.get(field)).lower() == str(value).lower()


def search_objects(object_type: str, field: str = "", value: str = "") -> dict:
    """Find objects of one type, optionally where `field` equals `value`.
    object_type is one of: Supplier, Product, Store, Inventory, PurchaseOrder."""
    _log("search_objects", object_type=object_type, field=field, value=value)
    if object_type not in DB:
        return {"error": f"Unknown type. Valid: {list(DB)}"}
    return {"results": [o for o in DB[object_type].values()
                        if not field or _match(o, field, value)]}


def get_linked_objects(object_type: str, object_id: str, link: str) -> dict:
    """Follow a link from one object to related objects, for example
    object_type='Product', object_id='P1', link='inventory'."""
    _log("get_linked_objects", object_type=object_type, object_id=object_id, link=link)
    spec = LINKS.get((object_type, link))
    if not spec:
        valid = [l for (t, l) in LINKS if t == object_type]
        return {"error": f"No link '{link}' on {object_type}. Valid: {valid}"}
    to_type, fk, kind = spec
    obj = DB.get(object_type, {}).get(object_id)
    if not obj:
        return {"error": f"{object_type} {object_id} not found"}
    if kind == "one":
        target = DB[to_type].get(obj[fk])
        return {"results": [target] if target else []}
    return {"results": [o for o in DB[to_type].values() if o.get(fk) == object_id]}


def find_low_stock() -> dict:
    """Return all Inventory objects where quantity is below the reorder point."""
    _log("find_low_stock")
    return {"results": [i for i in DB["Inventory"].values()
                        if i["quantity"] < i["reorder_point"]]}


def execute_action(action: str, params_json: str) -> dict:
    """Execute an ontology action. params_json is a JSON string of the action's
    parameters, e.g. '{"product_id": "P1", "quantity": 50}'. A human must approve."""
    _log("execute_action", action=action, params_json=params_json)
    return run_action(action, params_json, approve=human_approval)


def run_action(action, params_json, approve):
    spec = ACTIONS.get(action)
    if not spec:
        return {"error": f"Unknown action. Valid: {list(ACTIONS)}"}
    try:
        params = json.loads(params_json) if isinstance(params_json, str) else dict(params_json)
    except (json.JSONDecodeError, TypeError, ValueError):
        return {"error": "params_json must be a valid JSON object"}
    if not approve(action, params):
        return {"status": "rejected_by_human"}
    try:
        return {"status": "ok", "result": spec["fn"](**params)}
    except (ValueError, TypeError) as e:
        return {"error": str(e)}


TOOLS = [search_objects, get_linked_objects, find_low_stock, execute_action]


def describe_ontology():
    return {
        "object_types": OBJECT_FIELDS,
        "links": [f"{t}.{l} -> {v[0]} ({v[2]})" for (t, l), v in LINKS.items()],
        "actions": {n: {"description": a["description"], "params": a["params"]}
                    for n, a in ACTIONS.items()},
    }


SYSTEM = f"""You are an operations agent for a retail chain. Today is 2026-10-05.
You do NOT know the business data; always query it with tools. Never invent
products, stores or numbers. Change data only through execute_action.
For multi-hop questions, search first, then follow links.
Before proposing a purchase order or transfer, check stock in other stores:
moving existing stock is usually cheaper than ordering.

ONTOLOGY:
{json.dumps(describe_ontology(), indent=2)}
"""


# ---------------------------------------------------------------- 4. AGENT
def get_api_key():
    key = os.environ.get("GEMINI_API_KEY", "")
    if not key:
        try:  # Colab secrets
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = ""
    if not key:
        from getpass import getpass
        key = getpass("Paste your Gemini API key (input hidden): ")
    key = key.strip().strip("'\"")
    if not key.startswith(("AIza", "AQ.")):  # AI Studio now issues AQ. keys; older ones start AIza
        sys.exit("That does not look like an AI Studio key (should start with 'AQ.' or 'AIza'). "
                 "Create one at https://aistudio.google.com/apikey")
    return key


def friendly(e):
    code = getattr(e, "code", None)
    hints = {
        401: "Key rejected. Run: pip install -U google-genai (older SDKs reject AQ. keys), then retry. If it persists, create a new key at aistudio.google.com/apikey (not a Vertex AI / Google Cloud express key).",
        403: "Access denied. The key may be restricted or the API not enabled for it.",
        404: "Model name not found. Set GEMINI_MODEL to a current model from AI Studio.",
        429: "Rate limit or quota reached. Wait a minute and try again.",
    }
    return f"Gemini error {code}: {hints.get(code, str(e)[:300])}"


RETRYABLE = (429, 500, 502, 503, 504)  # temporary Google-side problems
FALLBACK_MODELS = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-2.0-flash", "gemini-flash-lite-latest"]


def pick_models(client):
    """Return every usable model name, best first (skips names that return 404)."""
    from google.genai import errors
    import time
    wanted = os.environ.get("GEMINI_MODEL")
    candidates = [wanted] if wanted else FALLBACK_MODELS
    usable = []
    for m in candidates:
        for attempt in range(3):
            try:
                client.models.generate_content(model=m, contents="Reply with the word OK.")
                usable.append(m)
                break
            except errors.APIError as e:
                code = getattr(e, "code", None)
                if code == 404:
                    break  # model name does not exist, skip it
                if code in RETRYABLE:
                    if attempt == 2:
                        usable.append(m)  # exists but busy; keep it as an option
                        break
                    time.sleep(2 * (attempt + 1))
                    continue
                sys.exit(friendly(e))
    if not usable:
        sys.exit("No working Gemini model found. Set GEMINI_MODEL to a model listed in AI Studio.")
    return usable


def ask(client, types, models, state, question):
    """Send a question; retry busy errors with waiting, then fall back to other models."""
    from google.genai import errors
    import time
    new_chat = lambda m: client.chats.create(model=m, config=types.GenerateContentConfig(
        system_instruction=SYSTEM, tools=TOOLS))
    if state.get("chat") is None:
        state["chat"], state["model"] = new_chat(models[0]), models[0]
    order = [state["model"]] + [m for m in models if m != state["model"]]
    last = None
    for m in order:
        if m != state["model"]:
            print(f"  (switching to model {m}; earlier chat memory is reset)")
            state["chat"], state["model"] = new_chat(m), m
        for attempt in range(4):
            try:
                return state["chat"].send_message(question).text or "(no text reply)"
            except errors.APIError as e:
                last = e
                if getattr(e, "code", None) not in RETRYABLE:
                    return friendly(e)
                wait = 3 * (attempt + 1)
                print(f"  (Google says the model is busy; retrying in {wait}s...)")
                time.sleep(wait)
    return friendly(last) + "\nAll models are busy right now. Wait a few minutes and try again."


def agent():
    global VERBOSE
    VERBOSE = True
    try:
        from google import genai
        from google.genai import types
    except ImportError:
        sys.exit("Install the SDK first:  pip install -U google-genai")

    os.environ.pop("GOOGLE_API_KEY", None)  # stop the SDK preferring another key
    client = genai.Client(api_key=get_api_key())
    models = pick_models(client)
    state = {"chat": None, "model": None}

    print(f"Retail agent ready (models: {', '.join(models)}).")
    print("Try: 'Which products are low on stock, and what should we do?'  Type 'quit' to exit.\n")
    while True:
        try:
            q = input("> ").strip()
        except EOFError:
            break
        if q.lower() in ("quit", "exit", ""):
            break
        print("\n" + ask(client, types, models, state, q) + "\n")


def selftest():
    no = lambda *a: False
    yes = lambda *a: True
    print("Low stock:", find_low_stock())
    print("Rice in all stores:", get_linked_objects("Product", "P1", "inventory"))
    print("Supplier of P2:", get_linked_objects("Product", "P2", "supplier"))
    print("Electronics:", search_objects("Product", "category", "electronics"))
    print("Order rejected:", run_action("create_purchase_order",
          '{"product_id": "P3", "quantity": 50}', no))
    print("Order approved:", run_action("create_purchase_order",
          '{"product_id": "P3", "quantity": 50}', yes))
    print("Transfer rice ST2->ST1:", run_action("transfer_stock",
          '{"product_id": "P1", "from_store_id": "ST2", "to_store_id": "ST1", "quantity": 30}', yes))
    print("Bad markdown (80%):", run_action("apply_markdown",
          '{"product_id": "P4", "percent": 80}', yes))
    print("Markdown 20%:", run_action("apply_markdown",
          '{"product_id": "P4", "percent": 20}', yes))
    print("Bad JSON:", run_action("apply_markdown", "not json", yes))


if __name__ == "__main__":
    selftest() if "--selftest" in sys.argv else agent()

Retail agent ready (models: gemini-flash-latest, gemini-flash-lite-latest).
Try: 'Which products are low on stock, and what should we do?'  Type 'quit' to exit.

> Which products are low on stock, and what should we do?
  (Google says the model is busy; retrying in 3s...)
  (Google says the model is busy; retrying in 6s...)
  (Google says the model is busy; retrying in 9s...)
  tool> find_low_stock({})
  (Google says the model is busy; retrying in 12s...)
  (switching to model gemini-flash-lite-latest; earlier chat memory is reset)
  tool> find_low_stock({})
  tool> search_objects({"object_type": "Inventory", "field": "", "value": ""})
  tool> search_objects({"object_type": "Product", "field": "", "value": ""})
  tool> search_objects({"object_type": "Store", "field": "", "value": ""})
  tool> search_objects({"object_type": "Supplier", "field": "", "value": ""})

We have 3 low stock inventory items across our stores. Per company policy, we should check other stores for excess stock befo